In [1]:
import chromadb
import sentence_transformers
import rank_bm25
import tiktoken
import frontmatter ## used reading 156 files to extract metadata and content put it into one single file called corpus.jsonl
import markdown_it
print("All Phase 1 imports OK")

All Phase 1 imports OK


# Cell 3  Ingest

## Purpose

Read all 156 FastAPI documentation files, clean MDX build syntax from their content, extract metadata + cleaned text from each one, and write everything into a single structured index file called corpus.jsonl.

Working with 156 files individually is really hard  you have to loop over them every time, and loading them for further steps (e.g., embedding, chunking) can be hard. The solution is to put the metadata and content into a single file.

The second reason is that the docs files are not clean  they use MDX, which is not human-readable content. For example:

| Junk | Example | What it is |
|---|---|---|
| Anchor suffix | { #query-parameters } | Website heading anchor |
| Include directive | {* ../../docs_src/query_params/tutorial001.py hl[9] *} | Build-time code snippet include |

These things should be removed because:
- Vectors get polluted with build-path noise
- Retrieval gets confused by ../../docs_src/... strings
- Citations show ugly { #anchor } artifacts

## Input

- *Source:* D:\dev-docs-assistant\fastapi\docs\en\docs\
- *Files:* 156 .md files (17.69 MB)
- *Format:* MDX (Markdown + JSX build syntax)

## Output

- *File:* D:\dev-docs-assistant\data\processed\corpus.jsonl
- *Format:* JSON Lines — one JSON object per line
- *Records:* ~153 (meta folders skipped, near-empty pages dropped)

## The cleaning function

```python
def clean_text(text):
    # 1. Remove MDX anchor suffixes: { #anchor-id }
    text = re.sub(r"\{[^}]*\}", "", text)
    # 2. Remove MDX include directives: {* path *}
    text = re.sub(r"\{\.?\*\}", "", text, flags=re.DOTALL)
    # 3. Trim trailing whitespace on lines
    text = re.sub(r"[ \t]+\n", "\n", text)
    # 4. Collapse 3+ blank lines to 2
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

In [22]:
# Import required libraries
from pathlib import Path
import json
import re
import frontmatter


# Define input and output paths
ROOT = Path(r"D:\dev-docs-assistant\fastapi\docs\en\docs")
OUTPUT = Path(r"D:\dev-docs-assistant\data\processed\corpus.jsonl")

# Documentation URL and folders to skip
BASE_URL = "https://fastapi.tiangolo.com/"
SKIP = {"about", "resources", "css", "img", "js"}


# Clean MDX syntax from Markdown content
def clean_text(text):

    # Remove MDX include directives
    text = re.sub(r"\{\.?\*\}", "", text, flags=re.DOTALL)

    # Remove heading anchors
    text = re.sub(r"\{[^}]*\}", "", text)

    # Remove trailing whitespace from lines
    text = re.sub(r"[ \t]+\n", "\n", text)

    # Remove excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


# Store processed documents
records = []

# Find and process all Markdown files
for file in ROOT.rglob("*.md"):

    # Get relative file path
    path = file.relative_to(ROOT)

    # Skip unwanted folders
    if any(folder in SKIP for folder in path.parts[:-1]):
        continue

    # Read file and extract metadata
    raw = file.read_text(encoding="utf-8")
    doc = frontmatter.loads(raw)

    # Clean Markdown content
    content = clean_text(doc.content)

    # Skip documents shorter than 100 characters
    if len(content) < 100:
        continue

    # Extract the first H1 heading as the title
    match = re.search(r"(?m)^#\s+(.+)$", content)
    title = match.group(1).strip() if match else "Untitled"

    # Generate the public documentation URL
    url_path = path.with_suffix("")

    if url_path.name == "index":
        url_path = url_path.parent

    # Store document information in a dictionary
    records.append({
        "doc_id": "__".join(path.with_suffix("").parts),
        "source_path": path.as_posix(),
        "url": BASE_URL + "/".join(url_path.parts) + "/",
        "title": title,
        "metadata": dict(doc.metadata),
        "text": content,
        "char_count": len(content)
    })


# Create output directory if it doesn't exist
OUTPUT.parent.mkdir(parents=True, exist_ok=True)

# Write each document as one JSON line
with OUTPUT.open("w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


# Display processing summary
print(f"Processed {len(records)} documents.")
print(f"Saved to: {OUTPUT}")

Processed 153 documents.
Saved to: D:\dev-docs-assistant\data\processed\corpus.jsonl
